# Домашнее задание 2 (5 баллов).

*Все задания ниже имеют равный вес (5/10)*

Код для импорта мы написали за вас (не благодарите, нам не трудно). Дальше код будете писать вы. 

[Тут](https://habr.com/ru/companies/ruvds/articles/494720/) шпора по pandas. За основу домашнего задания взят ноутбук [отсюда](https://rutube.ru/video/f884aa6ed5f94120b7304506042fe5bb/) (не подглядывайте!).

In [1]:
import pandas as pd
import numpy as np

#### Описание данных

Автор д/з - плохой человек, который не стал переводить описание с мотивировкой, что весь DS на английском. Так что описание полей будет на английском:

1. Account ID
- Description: A unique identifier for each social media account in the dataset.
- Type: Integer
- Example: 1, 2, 3, …
2. Username
- Description: The username or handle of the social media account.
- Type: String
- Example: john_doe, tech_guru_22, fitness_freak
3. Platform
- Description: The social media platform the account is using (Instagram, Twitter, Facebook, TikTok, LinkedIn).
- Type: Categorical (String)
- Example: Instagram, Twitter, Facebook, TikTok, LinkedIn
4. Follower Count
- Description: The total number of followers the account has.
- Type: Integer
- Example: 1500, 245000, 78000
5. Posts Per Week
- Description: The average number of posts the account creates per week.
- Type: Integer
- Example: 3, 5, 7
6. Engagement Rate
- Description: The percentage of interactions (likes, comments, shares) relative to the follower count. This is a measure of how engaging the content is.
- Type: Float
- Range: 0.01 to 0.15
- Example: 0.045 (4.5% engagement rate)
7. Ad Spend (USD)
- Description: The monthly amount spent on advertising or promoting posts.
- Type: Float
- Example: 150.75, 850.00, 300.50
8. Conversion Rate
- Description: The percentage of users who take a desired action (e.g., clicking a link, signing up, etc.) after interacting with an ad.
- Type: Float
- Range: 0.01 to 0.05 (1% to 5% conversion rate)
- Example: 0.025 (2.5% conversion rate)
9. Campaign Reach
- Description: The total number of unique users reached by the user’s campaigns in a given month.
- Type: Integer
- Example: 5000, 20000, 15000

#### Задание 0

Подгрузите данные. Да-да, за чтение таблицы баллов не будет))

**Hint**: [pd.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)

In [2]:
df = pd.read_csv("data.csv", sep = ",")

In [3]:
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083


#### Задание 1

Колонка `Platform` содержит название различных платформ. Давайте представим, что в них есть некоторое отношение порядка. Закодируйте каждую платформу целым числом (от 0 до N) и положите этот "код" в новую колонку `Platform_Code`. Теперь вычислите корреляцию Спирмена между всеми парами колонок в датасете (результатом будет таблица корреляций). В качестве ответа выведите значение корреляции `Platform_Code` с `Engagement Rate`. Можете после вывода числа еще коротко написать, что оно означает (нет, это не оценивается).

**Hint**: [pd.factorize](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.factorize.html), [pd.DataFrame.select_dtypes](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.select_dtypes.html), [pd.DataFrame.corr](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html).

In [4]:
codes, uniques = pd.factorize(df["Platform"]) #перекодировка колонки "Platform"; возвращает кортеж из перекодированных и уникальных значений
df["Platform_Code"] = codes #создание нового столбца c перекодированными значениями столбца "Platform"
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2


In [5]:
#попарное вычисление корреляции Спирмена для числовых значений и вывод корреляции между "Platform_Code" и "Engagement Rate"
df.corr(numeric_only = True, method = "spearman").loc["Platform_Code", "Engagement Rate"] 

0.03138169529349812

In [6]:
codes, uniques = pd.factorize(df["Platform"])
df["Platform_Code"] = codes
df.select_dtypes(include = 'number').corr(method = "spearman").loc["Platform_Code", "Engagement Rate"]

0.03138169529349812

In [7]:
codes, uniques = pd.factorize(df["Platform"])
df["Platform_Code"] = codes
df.select_dtypes(exclude = 'object').corr(method = "spearman").loc["Platform_Code", "Engagement Rate"]

0.03138169529349812

#### Задание 2

Теперь посмотрите на столбец `Follower Count`. В нем какие-то числа. Иногда бывает полезно провести дискретизацию такого признака. Разбейте все значения в столбце на 4 группы: "Low", "Medium", "High", "Very High". Каждая группа включает в себя новые 25% данных. То есть, Low включает в себя 25% самых маленьких значений признака и так далее. Положите значения "Low", "Medium", "High" или "Very High" для каждого сэмпла датасета в новую колонку `Follower_Bin`. Теперь посчитайте среднее значение `Engagement Rate` для каждой категории из `Follower_Bin`. В качестве ответа выведите значение для категории "High".

**Hint**: [pd.qcut](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.qcut.html), [pd.groupby](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html), [pd.DataFrame.mean](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html)

In [8]:
df["Follower Count"].head()

0     54217
1    987518
2    218870
3    207432
4    350204
Name: Follower Count, dtype: int64

In [9]:
#разбиение значений столбца "Follower Count" на 4 группы в соответствии с квантилями
df["Follower_Bin"] = pd.qcut(df["Follower Count"], q = 4, labels = ["Low", "Medium", "High", "Very High"]) 
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium


In [10]:
#группировка значений в соответствии со значениями слобца "Follower_Bin", вычисление среднего значения параметра "Engagement Rate" в каждой группе
df.groupby("Follower_Bin", observed = True)["Engagement Rate"].mean()                                                               

Follower_Bin
Low          0.087032
Medium       0.086858
High         0.086550
Very High    0.086505
Name: Engagement Rate, dtype: float64

In [11]:
df.groupby("Follower_Bin", observed = True)["Engagement Rate"].mean()["High"] #вывод среднего значения параметра "Engagement Rate" для категории "High"

0.08655032

#### Задание 3

Иногда бывает полезно превратить широкую таблицу в длинную (например, для визуализаций сразу нескольких признаков на одной картинке). Да, звучит странно, но именно этим вы сейчас и займетесь. Сделайте новый датафрейм `melted_df`, в который вы поместите каждый сэмпл датасета 6 раз: по одному разу на значение из 'Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate' и 'Campaign Reach'. То есть, вы берете сэмпл из датасета (строку) и превращаете ее в 6 отдельных строк. Каждая отдельная строка в столбце `Metric` имеет имя из предложенного списка 5 признаков, а в столбце `Value` - значение данного сэмпла по этому признаку. Значение `Platform` повторяется в этих 6 строках.

Иначе говоря, 

```json
{
    "Account ID": 1,
    "Username": "harrislisa",
    "Platform": "TikTok",
    "Follower Count": 54217,
    "Posts Per Week": 3,
    "Engagement Rate": 0.0986,
    "Ad Spend (USD)": 538.1,
    "Conversion Rate": 0.049,
    "Campaign Reach": 1308,
    "Platform_Code": 0,
    "Follower_Bin": "Low"
}
```

превращается в 

```json
{
    "Platform": "TikTok",
    "Metric": "Follower Count",
    "Value": 54217,
},
{
    "Platform": "TikTok",
    "Metric": "Posts Per Week",
    "Value": 3,
}, ...
```

Для каждого уникальной пары значений (`Platform`, `Metric`) посчитайте моду среди всех значений `Value` для этой пары, результат сделайте списком и оставьте только наибольшее. В качестве ответа выведите сумму полученных мод (сумму всех значений в столбце `Value` уже после вычисления мод). Иначе говоря, выведите сумму всех мод значений для всех уникальных пар (`Platform`, `Metric`).

**Hint**: [pd.melt](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.melt.html), [pd.DataFrame.mode](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mode.html), [pd.DataFrameGroupBy.agg](https://pandas.pydata.org/docs/dev/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)

In [12]:
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium


In [13]:
#создание нового датафрейма, в котором по очереди перебираются метрики из списка
melted_df = df.melt(id_vars = "Platform", 
                    value_vars = ["Follower Count", "Posts Per Week", "Ad Spend (USD)", "Conversion Rate", "Engagement Rate", "Campaign Reach"], 
                    var_name = "Metric",
                    value_name = "Value")
melted_df.head()

,Platform,Metric,Value
0,TikTok,Follower Count,54217.0
1,LinkedIn,Follower Count,987518.0
2,Facebook,Follower Count,218870.0
3,Instagram,Follower Count,207432.0
4,Facebook,Follower Count,350204.0


In [14]:
#группировка в соответствии со значениями слобцов "Platform" и "Metric", вычисление наибольшей моды параметра "Value" в каждой группе
modes = melted_df.groupby(["Platform", "Metric"])[["Value"]].agg(lambda x: x.mode().max())["Value"] 
modes

Platform   Metric         
Facebook   Ad Spend (USD)        421.5600
           Campaign Reach      49860.0000
           Conversion Rate         0.0186
           Engagement Rate         0.0856
           Follower Count     350858.0000
           Posts Per Week          5.0000
Instagram  Ad Spend (USD)        878.6800
           Campaign Reach      48629.0000
           Conversion Rate         0.0274
           Engagement Rate         0.0986
           Follower Count     999726.0000
           Posts Per Week          5.0000
LinkedIn   Ad Spend (USD)        783.6000
           Campaign Reach      14912.0000
           Conversion Rate         0.0186
           Engagement Rate         0.0642
           Follower Count      92848.0000
           Posts Per Week          6.0000
TikTok     Ad Spend (USD)        951.6500
           Campaign Reach      48053.0000
           Conversion Rate         0.0227
           Engagement Rate         0.0856
           Follower Count     700758.0000
       

In [15]:
np.sum(modes) #вычисление суммы всех мод

3100285.4716

#### Задание 4

А теперь хочется посмотреть на самые популярные аккаунты на разных платформах. Для каждой платформы отсортируйте датафрейм по убыванию количества подписчиков (`Follower Count`) - да, без циклов, сразу для всех платформ сделать сортировку, а затем оставьте только первые три записи для каждой платформы - это и будут три самых популярных аккаунта для каждой платформы. В качестве ответа выведите саму таблицу и минимальное значение `Follower Count` в ней.

**Hint**: к *groupby* можно применять функции - это эквивалентно применению функции к каждой "группе" внутри groupby-объекта. Читайте [про применение apply к датафрейму после groupby](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#flexible-apply).

In [16]:
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium


In [17]:
#группировка значений в соответствии со значениями слобца "Platform", применение сортировки по убыванию подписчиков и вывод 3 наиболее популярных аккаунтов на каждой из платформ
df.groupby("Platform").apply(lambda x: x.sort_values(by = "Follower Count", ascending = False).head(3)) 

Account ID         Username   Platform  Follower Count  \
Platform                                                                 
Facebook  2403        2404           eric65   Facebook          999982   
          7350        7351     patricknoble   Facebook          997915   
          1689        1690      chavezjason   Facebook          997512   
Instagram 8685        8686  alexandersamuel  Instagram          999726   
          3965        3966         lrodgers  Instagram          999351   
          2189        2190           jbrown  Instagram          997844   
LinkedIn  3039        3040          toneill   LinkedIn          999055   
          6359        6360    andrewgregory   LinkedIn          998968   
          2159        2160     ashleycooper   LinkedIn          998925   
TikTok    5838        5839     edwardthomas     TikTok          999739   
          4234        4235    andradewesley     TikTok          999234   
          2575        2576     williamwyatt     TikTok          998623   
Twitter   4920        4921      teresaellis    Twitter          999919   
          9684        9685           sriley    Twitter          999442   
          7576        7577       peggymunoz    Twitter          998216   

                Posts Per Week  Engagement Rate  Ad Spend (USD)  \
Platform                                                          
Facebook  2403               6           0.0642          884.06   
          7350               3           0.0834          429.01   
          1689               7           0.0834          993.20   
Instagram 8685               3           0.0834          687.61   
          3965               1           0.0834          565.07   
          2189               5           0.0642          505.61   
LinkedIn  3039               4           0.0642          799.49   
          6359               7           0.1020          797.64   
          2159               6           0.0856          474.46   
TikTok    5838               7           0.0642          630.77   
          4234               5           0.0834          872.77   
          2575               6           0.0856          477.98   
Twitter   4920               6           0.0834          411.63   
          9684               3           0.0834          206.84   
          7576               6           0.0642          456.61   

                Conversion Rate  Campaign Reach  Platform_Code Follower_Bin  
Platform                                                                     
Facebook  2403           0.0281           17312              2    Very High  
          7350           0.0182           25985              2    Very High  
          1689           0.0397           45717              2    Very High  
Instagram 8685           0.0205           11050              3    Very High  
          3965           0.0335           12391              3    Very High  
          2189           0.0202           14717              3    Very High  
LinkedIn  3039           0.0174           21862              1    Very High  
          6359           0.0351           15552              1    Very High  
          2159           0.0156           45956              1    Very High  
TikTok    5838           0.0325           35523              0    Very High  
          4234           0.0481           17188              0    Very High  
          2575           0.0250           43299              0    Very High  
Twitter   4920           0.0460            3975              4    Very High  
          9684           0.0225           12783              4    Very High  
          7576           0.0456           22037              4    Very High

In [18]:
#вывод минимального значения в столбце "Follower Count"
df.groupby("Platform").apply(lambda x: x.sort_values(by = "Follower Count", ascending = False).head(3))["Follower Count"].min()  

997512

#### Задание 5

Хочется посчитать какую-то метрику. Мы хотим посмотреть, на отношение разности суммы подписчиков аккаунтов с высокой и низкой конверсией к суммарному охвату рекламы на каждой платформе. То есть, мы делим аккаунты на две группы: высокая и низка конверсия. Затем мы смотрим на то, на сколько сильно влияние аккаунтов с высокой конверсией по сравнению с аккаунтами с низкой конверсией. 

Давайте определим *Conversion Influence* следущим образом:

$$Conversion Influence = \frac{Total Follower\ Count (High) - Total Follower\ Count (Low)}{Total Campaign Reach (High)+Total Campaign Reach (Low)}$$

Считать эту метрику мы будет для каждой `Platform`. В этой формуле High - это значения всех сэмплов датасета, в которых `Conversion Rate` больше медианы, а `Low` - не более медианы. `Total Feature` - это суммарное количество значений `Feature` либо по `High` сэмплам, либо по `Low`.

Чтобы постоянно не пересчитывать, где High. где Low, сделайте новую колонку в датасете `Conversion_Category`. Положите в нее для каждой строки либо High, либо Low.

Выведите платформу с самым большим `Conversion Influence`.

**Hint**: данное задание не про *groupby*, а скорее про [pd.pivot_table](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html). Сделайте сводную таблицу, по которой уже можно посчитать суммы, а затем подставить их в формулы.

In [19]:
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium


In [20]:
med = df["Conversion Rate"].median() #вычисление медианного значения параметра "Conversion Rate"
df["Conversion_Category"] = np.where(df["Conversion Rate"] > med, "High", "Low") #создание столбца "Conversion_Category" в соответствии с условием
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin,Conversion_Category
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low,High
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High,Low
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low,High
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low,High
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium,High


In [21]:
#создание сводной таблицы со значениями "Total Feature"
table = df.pivot_table(index = "Platform", columns = "Conversion_Category", values = ["Follower Count", "Campaign Reach"], aggfunc = "sum")
#создание столбца "Conversion Influence" для каждой из платформ
table["Conversion Influence"] = (table["Follower Count"]["High"] - table["Follower Count"]["Low"]) / (table["Campaign Reach"]["High"] + table["Campaign Reach"]["Low"])
table

Campaign Reach           Follower Count             \
Conversion_Category           High       Low           High        Low   
Platform                                                                 
Facebook                  26096572  25635749      512439819  499712373   
Instagram                 24250993  26482804      487970573  524726855   
LinkedIn                  25186827  25147594      509450797  499766430   
TikTok                    24253556  25091283      495899696  491110679   
Twitter                   26376400  24566705      536955379  481382949   

                    Conversion Influence  
Conversion_Category                       
Platform                                  
Facebook                        0.246025  
Instagram                      -0.724493  
LinkedIn                        0.192400  
TikTok                          0.097052  
Twitter                         1.090872

In [22]:
conv = pd.DataFrame(table["Conversion Influence"]) #создание датафрейма со значениями "Conversion Influence"
conv

,Conversion Influence
Platform,
Facebook,0.246025
Instagram,-0.724493
LinkedIn,0.192400
TikTok,0.097052
Twitter,1.090872


In [23]:
conv["Conversion Influence"].idxmax() #вывод названия платформы с наибольшим значением показателя "Conversion Influence"

'Twitter'

#### Задание 6

Мы знаем, что вам понравилось считать метрики по формуле. Давайте закрепим этот успех. Теперь для каждой платформы посчитаем, на сколько эффективна реклама в разрезе трех последовательных записей в датасете. 

Для каждой платформы отсортируйте записи в порядке убывания `Posts Per Week`. Будто бы аккаунты, которые постят чаще, используют более "активные" стратегии по рекламе. Теперь посчитайте *скользущие суммы с окном 3* по `Campaign Reach` и `Ad Spend (USD)`. Скользящая сумма с окном N - это вы идете по массиву, берете все последовательные тройки записей и суммируете их. Для первых двух записей троек не найдется. Для них скользящее среднее - NaN, что нам не помешает. 

Теперь для каждого окна посчитайте 

$$Rolling Efficiency Ratio = \frac{Rolling Sum of Campaign Reach}{Rolling Sum of Ad Spend}$$

По сути, для каждого окна вы посчитаете сколько пользователе привлеклось за один доллар, потреченный на рекламу, в данном окне. Понятно, что значений будет столько, сколько окон. Нам интересно максимально значение такой эффективности для каждой платформы.

В качестве ответа выведите название платформы с наибольшей максимальной эффективность и наименьшей (два названия, не одно, не три, ровно два).

**Hint**: окна можно делать через [pd.DataFrame.rolling](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html).

In [24]:
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin,Conversion_Category
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low,High
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High,Low
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low,High
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low,High
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium,High


In [25]:
#группировка значений в соответствии со значениями слобца "Platform", применение сортировки по убыванию значения параметра "Posts Per Week"
df.groupby("Platform").apply(lambda x: x.sort_values(by = "Posts Per Week", ascending = False)) 

Account ID          Username  Platform  Follower Count  \
Platform                                                                
Facebook 2841        2842          kristy65  Facebook          843888   
         3995        3996            anne79  Facebook          234438   
         1090        1091           sarah18  Facebook          519602   
         3770        3771      justinwilson  Facebook          854927   
         7365        7366            ghouse  Facebook           14727   
...                   ...               ...       ...             ...   
Twitter  6833        6834  jonathanreynolds   Twitter          840585   
         6820        6821        masonjames   Twitter           82238   
         6795        6796          nroberts   Twitter          953147   
         6747        6748         jessica26   Twitter          397430   
         6787        6788           emily77   Twitter          498712   

               Posts Per Week  Engagement Rate  Ad Spend (USD)  \
Platform                                                         
Facebook 2841               7           0.0856           62.71   
         3995               7           0.1020          441.13   
         1090               7           0.0856          494.50   
         3770               7           0.0856          353.04   
         7365               7           0.0856          548.50   
...                       ...              ...             ...   
Twitter  6833               1           0.1020          375.55   
         6820               1           0.0986          612.01   
         6795               1           0.0834          474.87   
         6747               1           0.0642          573.41   
         6787               1           0.1020          264.42   

               Conversion Rate  Campaign Reach  Platform_Code Follower_Bin  \
Platform                                                                     
Facebook 2841           0.0405           34007              2    Very High   
         3995           0.0445           39173              2          Low   
         1090           0.0255            9884              2         High   
         3770           0.0237           49583              2    Very High   
         7365           0.0138           20931              2          Low   
...                        ...             ...            ...          ...   
Twitter  6833           0.0181           45975              4    Very High   
         6820           0.0174           12703              4          Low   
         6795           0.0478           36164              4    Very High   
         6747           0.0334           48858              4       Medium   
         6787           0.0207           24760              4       Medium   

              Conversion_Category  
Platform                           
Facebook 2841                High  
         3995                High  
         1090                 Low  
         3770                 Low  
         7365                 Low  
...                           ...  
Twitter  6833                 Low  
         6820                 Low  
         6795                High  
         6747                High  
         6787                 Low  

[10000 rows x 12 columns]

In [26]:
#расчет скользущих суммы с окном 3 по параметру "Campaign Reach" 
df["Rolling Sum of Campaign Reach"] = df["Campaign Reach"].rolling(3).sum() 
#расчет скользущих суммы с окном 3 по параметру "Ad Spend (USD)" 
df["Rolling Sum of AD Spend"] = df["Ad Spend (USD)"].rolling(3).sum()
#вычисление параметра "Rolling Efficiency Ratio"
df["Rolling Efficiency Ratio"] = df["Rolling Sum of Campaign Reach"] / df["Rolling Sum of AD Spend"]
df

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin,Conversion_Category,Rolling Sum of Campaign Reach,Rolling Sum of AD Spend,Rolling Efficiency Ratio
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low,High,NaN,NaN,NaN
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High,Low,NaN,NaN,NaN
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low,High,25653.0,1167.70,21.968828
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low,High,36419.0,1562.22,23.312338
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium,High,37200.0,1587.42,23.434252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,sbyrd,TikTok,388169,6,0.0642,362.91,0.0108,45091,0,Medium,Low,117788.0,1494.08,78.836475
9996,9997,jacksongerald,TikTok,150775,1,0.0642,177.92,0.0262,33084,0,Low,Low,113007.0,884.97,127.695854
9997,9998,eryan,Twitter,427451,5,0.0834,432.65,0.0402,12484,4,Medium,High,90659.0,973.48,93.128775
9998,9999,barbara57,LinkedIn,132884,4,0.0986,892.99,0.0329,35766,1,Low,High,81334.0,1503.56,54.094283


In [27]:
#создание датафрейма с максимальными показателями "Rolling Efficiency Ratio" для каждой из платформ
rer = pd.DataFrame(df.groupby("Platform")["Rolling Efficiency Ratio"].apply(lambda x: x.max(numeric_only = True)))
rer

,Rolling Efficiency Ratio
Platform,
Facebook,328.530620
Instagram,285.002563
LinkedIn,389.502077
TikTok,301.714541
Twitter,421.765637


In [28]:
#вывод названия платформы с наибольшей максимальной эффективностью и с наименьшей
rer["Rolling Efficiency Ratio"].idxmax(), rer["Rolling Efficiency Ratio"].idxmin()

('Twitter', 'Instagram')

In [3]:
#ПРАВИЛЬНОЕ РЕШЕНИЕ:

df5 = df.sort_values(by=['Platform', 'Posts Per Week'], ascending=False) # сортируем записи в порядке убывания `Posts Per Week` + добавляем Platform, чтобы объединить записи по платформам

rolling_sums = df5[['Campaign Reach', 'Ad Spend (USD)']].rolling(window=3).sum() # вычисляем скользящую сумму

df5[['Rolling Sum of Campaign Reach', 'Rolling Sum of AdSpend']] = rolling_sums[['Campaign Reach', 'Ad Spend (USD)']] # добавляем значения скользящей суммы в основную таблицу

df5['Rolling Efficiency Ratio'] = df5['Rolling Sum of Campaign Reach']/df5['Rolling Sum of AdSpend'] # вычисляем Rolling Efficiency Ratio

df6 = pd.pivot_table(df5, index=['Platform'], values=['Rolling Efficiency Ratio'], aggfunc='max').sort_values(by='Rolling Efficiency Ratio') # создаем сводную таблицу с максимальными значениями для каждой платформы -> 
# заранее сортируем ее -> выводим масимальное и минимальное из этих значений

print(df6.index[-1], df6.index[0]) 

Facebook TikTok


#### Задание 7

Это еще не все прекрасные функции pandas, которые мы хотим вам показать. Теперь вы посчитаете, сколько аккаунтов на каждой платформе одновременно лучшие по `Engagement Rate` и `Conversion Rate`.

Сделайте два отдельных суб-сета. В одном оставьте для каждой платфмормы один топовый аккаунт по `Engagement Rate`, в другом - по `Conversion Rate`. Соедините эти два подмножества по столбцу `Platform` так, что в одно строке есть описание сразу двух аккаунтов-лидеров. Теперь посмотрите равны ли имена аккаунтов в одной строке. Выведите количество строк, в которых названия аккаунтов совпадают.

In [29]:
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin,Conversion_Category,Rolling Sum of Campaign Reach,Rolling Sum of AD Spend,Rolling Efficiency Ratio
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low,High,NaN,NaN,NaN
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High,Low,NaN,NaN,NaN
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low,High,25653.0,1167.70,21.968828
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low,High,36419.0,1562.22,23.312338
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium,High,37200.0,1587.42,23.434252


In [30]:
#группировка значений в соответствии со значениями слобца "Platform", применение сортировки по убыванию значения 
#параметра "Engagement Rate" и вывод одного топового ак
df1 = df.groupby("Platform").apply(lambda x: x.sort_values(by = "Engagement Rate", ascending = False).head(1)) 
df1 = df1.drop("Platform", axis=1) #удаление столбца "Platform"
df1

,,Account ID,Username,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin,Conversion_Category,Rolling Sum of Campaign Reach,Rolling Sum of AD Spend,Rolling Efficiency Ratio
Platform,,,,,,,,,,,,,,,
Facebook,2,3,qthomas,218870,3,0.102,150.36,0.0318,11043,2,Low,High,25653.0,1167.70,21.968828
Instagram,8405,8406,toddbryan,839593,6,0.102,870.78,0.0395,10507,3,Very High,High,73486.0,1898.85,38.700266
LinkedIn,5023,5024,rgarrison,242256,7,0.102,902.41,0.0143,16017,1,Low,Low,60839.0,2080.13,29.247691
TikTok,7835,7836,allison03,457403,5,0.102,893.56,0.0145,3610,0,Medium,Low,62957.0,1736.46,36.255946
Twitter,4861,4862,mckenzieadam,736784,5,0.102,265.20,0.0481,36249,4,High,High,99868.0,1803.97,55.360122


In [31]:
#группировка значений в соответствии со значениями слобца "Platform", применение сортировки по убыванию значения 
#параметра "Conversion Rate" и вывод одного топового аккаунта по этому показателю
df2 = df.groupby("Platform").apply(lambda x: x.sort_values(by = "Conversion Rate", ascending = False).head(1))
df2 = df2.drop("Platform", axis=1) #удаление столбца "Platform"
df2

,,Account ID,Username,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin,Conversion_Category,Rolling Sum of Campaign Reach,Rolling Sum of AD Spend,Rolling Efficiency Ratio
Platform,,,,,,,,,,,,,,,
Facebook,7641,7642,jaclyn79,793755,3,0.0986,867.62,0.05,9758,2,Very High,High,66438.0,2222.70,29.890674
Instagram,2863,2864,aprilwilliams,222951,4,0.0986,767.52,0.05,45634,3,Low,High,69453.0,1820.12,38.158473
LinkedIn,8016,8017,jeff87,816658,1,0.0856,986.41,0.05,22822,1,Very High,High,74983.0,2059.59,36.406761
TikTok,284,285,robertmorris,702687,1,0.0856,276.36,0.05,2250,0,High,High,73317.0,1653.22,44.348000
Twitter,8969,8970,wreed,250978,4,0.1020,89.54,0.05,25753,4,Low,High,42923.0,591.20,72.603180


In [32]:
#объединение 2 подмножеств по столбцу "Platform" (в данном случае парметр how не влияет на результат,  
#т.к. названия платформ совпадают
df_new = pd.merge(df1, df2, on="Platform", how="outer") 
df_new

,Account ID_x,Username_x,Follower Count_x,Posts Per Week_x,Engagement Rate_x,Ad Spend (USD)_x,Conversion Rate_x,Campaign Reach_x,Platform_Code_x,Follower_Bin_x,...,Engagement Rate_y,Ad Spend (USD)_y,Conversion Rate_y,Campaign Reach_y,Platform_Code_y,Follower_Bin_y,Conversion_Category_y,Rolling Sum of Campaign Reach_y,Rolling Sum of AD Spend_y,Rolling Efficiency Ratio_y
Platform,,,,,,,,,,,,,,,,,,,,,
Facebook,3,qthomas,218870,3,0.102,150.36,0.0318,11043,2,Low,...,0.0986,867.62,0.05,9758,2,Very High,High,66438.0,2222.70,29.890674
Instagram,8406,toddbryan,839593,6,0.102,870.78,0.0395,10507,3,Very High,...,0.0986,767.52,0.05,45634,3,Low,High,69453.0,1820.12,38.158473
LinkedIn,5024,rgarrison,242256,7,0.102,902.41,0.0143,16017,1,Low,...,0.0856,986.41,0.05,22822,1,Very High,High,74983.0,2059.59,36.406761
TikTok,7836,allison03,457403,5,0.102,893.56,0.0145,3610,0,Medium,...,0.0856,276.36,0.05,2250,0,High,High,73317.0,1653.22,44.348000
Twitter,4862,mckenzieadam,736784,5,0.102,265.20,0.0481,36249,4,High,...,0.1020,89.54,0.05,25753,4,Low,High,42923.0,591.20,72.603180


In [33]:
#создание столбца с булевыми значениями, отражающими равенство аккаунтов в одной строке
df_new["Account Equality"] = df_new["Username_x"] == df_new["Username_y"] 
#вычисление количества аккаунтов с одинаковыми названиями
df_new["Account Equality"].value_counts().get(True, 0) 

0

#### Задание 8

Давайте теперь что-то попроще сделаем. Например, посчитаем отношение суммарного количества подписчиков на аккаунтах с высокой конверсией к такой же сумме в аккаунтах с низкой конверсией (очевидно, для каждой платформы). По сути, мы просто хотим получить число, которое характеризует, на сколько сильно аккаунты с высокой конверсией "доминируют" над аккаунтами с низкой конверсией в плане количества подписчиков.

Высокой конверсией будем считать конверсию больше средней. Остальное - низкая. Посчитайте суммы подписчиков для каждой платформы, поделите одно на другое и выведите разницу между самым большим значением и самым маленьким, а также платформы, которые соотвутствуют этим значениям.

Используйте магическую команду `%%time`, чтобы замерить, сколько времени ушло на исполнение вашего pandas-скрипта.

In [34]:
df.head()

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin,Conversion_Category,Rolling Sum of Campaign Reach,Rolling Sum of AD Spend,Rolling Efficiency Ratio
0,1,harrislisa,TikTok,54217,3,0.0986,538.10,0.0490,1308,0,Low,High,NaN,NaN,NaN
1,2,rhicks,LinkedIn,987518,5,0.0834,479.24,0.0174,13302,1,Very High,Low,NaN,NaN,NaN
2,3,qthomas,Facebook,218870,3,0.1020,150.36,0.0318,11043,2,Low,High,25653.0,1167.70,21.968828
3,4,carlosholt,Instagram,207432,6,0.0834,932.62,0.0400,12074,3,Low,High,36419.0,1562.22,23.312338
4,5,parsonsashley,Facebook,350204,2,0.0642,504.44,0.0463,14083,2,Medium,High,37200.0,1587.42,23.434252


In [35]:
%%time
#вычисление среднего значения параметра "Conversion Rate"
m = df["Conversion Rate"].mean() 
#создание столбца "Conversion_Category (Based on Mean)" в соответствии с условием
df["Conversion_Category (Based on Mean)"] = np.where(df["Conversion Rate"] > m, "High", "Low") 
#создание сводной таблицы с суммарными значениями подписчиков по группам
dfNew = df.pivot_table(index = "Platform", columns = "Conversion_Category (Based on Mean)", values = "Follower Count", aggfunc = "sum")
#создание столбца "Dominance" для каждой из платформ
dfNew["Dominance"] = dfNew["High"] / dfNew["Low"]
#вычисление разницы между максимальным и минимальным значениями
dfNew["Dominance"].max() - dfNew["Dominance"].min(), dfNew["Dominance"].idxmax(), dfNew["Dominance"].idxmin()

CPU times: total: 31.2 ms
Wall time: 21.7 ms


(0.17688741338715763, 'Twitter', 'Instagram')

#### Задание 9

А теперь решите задание 8 чисто питоном. Никаких функций и методов pandas. Только питоновские циклы. Замерьте время выполнения кода. Наконец, сравните время в задании 8 и 9. Напишите ниже, кто же победил: чистый python и pandas?

**Hint**: Чтобы итерироваться по датафрейму, можно из него сделать генератор через [pd.DataFrame.iterrows](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iterrows.html) или [pd.DataFrame.itertuples](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.itertuples.html#pandas.DataFrame.itertuples). К слову, это не все способы итерироваться по датафрейму.

In [36]:
dfNew

Conversion_Category (Based on Mean),High,Low,Dominance
Platform,,,
Facebook,491799506,520352686,0.945127
Instagram,468232352,544465076,0.859986
LinkedIn,490706904,518510323,0.946378
TikTok,482930355,504080020,0.958043
Twitter,518386631,499951697,1.036873


In [37]:
%%time

d = df.to_dict()
for elem in d['Account ID']:
    if d['Conversion Rate'][elem] > sum(d['Conversion Rate'].values()) / len(d['Conversion Rate'].values()):
        d['Conversion_Category (Based on Mean)'][elem] = 'High'
    else:
        d['Conversion_Category (Based on Mean)'][elem] = 'Low'

dNew = {}
for pl in set(d['Platform'].values()):
    dNew[pl] = {'High': 0, 'Low':0}

for ind in d['Account ID'].keys():
    if d['Conversion_Category (Based on Mean)'][ind] == 'High':
        key = d['Platform'][ind]
        dNew[key]['High'] += d['Follower Count'][ind]
    elif d['Conversion_Category (Based on Mean)'][ind] == 'Low':
        key = d['Platform'][ind]
        dNew[key]['Low'] += d['Follower Count'][ind]

dominances = {}
for pl in dNew:
    dominances[pl] = dNew[pl]['High'] / dNew[pl]['Low']

diff = max(dominances.values()) - min(dominances.values())

for pl, v in dominances.items():
    if v == max(dominances.values()):
        pl_max = pl
    elif v == min(dominances.values()):
        pl_min = pl

diff, pl_max, pl_min

CPU times: total: 828 ms
Wall time: 997 ms


(0.17688741338715763, 'Twitter', 'Instagram')

**А победителем является**: pandas

#### Задание 10

Крайне серьезное задание. Отнеситесь к нему соответствующе. В ячейке ниже напишите ваш любимый анекдот или мем (только без баянов, окей?). Можно плохие. Помните, это задание на полный балл. Проверяющий работу ассистент должен улыбнуться.

Если вставляете картинку, то убедитесь, что вы ее не подгружаете локально. А то будет неудобно - потерять балл на этом задании, когда надо было выложить картинку на облако и прокинуть ссылку. И нет, нельзя сюда просто ссылку вставить. Либо ищите, как вставить картинку, либо смешной анекдот. Есть всего два стула - выбирайте...

In [38]:
#У царя было косоглазие. Пошел он, куда глаза глядят, и разорвался :)